# MainCrafts Technology
## Artificial Intelligence & Machine Learning – Task 1

# Build & Evaluate a Linear Regression Model
### California Housing Price Predictor

**Intern:** Siddeshwar H  
**Task:** Task 1  
**Domain:** Artificial Intelligence & Machine Learning  
**Model:** Linear Regression  
**Dataset:** California Housing Dataset

---

### Objective
Build and evaluate a Linear Regression model using the California Housing dataset. This notebook covers the complete ML workflow required in the internship task:

1. Data loading
2. Data exploration and preprocessing
3. Exploratory Data Analysis (EDA)
4. Train/test split
5. Linear Regression model training
6. Evaluation using MAE, RMSE and R²
7. Actual vs predicted visualization
8. Residual analysis
9. Feature coefficient analysis
10. Model saving
11. Example prediction
12. Final findings and improvement ideas

**Submission note:** Run the notebook from top to bottom before submitting so that all outputs and graphs are visible.


## 1. Install / Import Required Libraries

If the libraries are not installed, use the following command in a terminal:

`pip install pandas numpy scikit-learn matplotlib seaborn jupyter`

The code below only imports the libraries; it does not reinstall packages every time the notebook runs.

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

print("All required libraries imported successfully.")


## 2. Load the California Housing Dataset

The California Housing dataset is available directly through scikit-learn. The first run may require an internet connection because scikit-learn downloads the dataset if it is not already cached.

In [ ]:
data = fetch_california_housing(as_frame=True)

# Combine the feature columns and target into one DataFrame
df = pd.concat(
    [data.data, data.target.rename("MedHouseVal")],
    axis=1
)

print("Dataset loaded successfully.")
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])
print("\nFirst 5 rows:")
display(df.head())


## 3. Understand the Dataset

### Features

| Feature | Meaning |
|---|---|
| MedInc | Median income |
| HouseAge | Median house age |
| AveRooms | Average number of rooms |
| AveBedrms | Average number of bedrooms |
| Population | Population of the district |
| AveOccup | Average household occupancy |
| Latitude | Geographic latitude |
| Longitude | Geographic longitude |

### Target

**MedHouseVal** is the median house value in a California district, represented in units of $100,000.


In [ ]:
print("Dataset information:")
df.info()

print("\nDataset shape:", df.shape)


In [ ]:
print("Descriptive statistics:")
display(df.describe().round(3))


## 4. Data Quality Check / Preprocessing

Before training, we check data types, missing values and duplicate rows.

In [ ]:
print("Missing values in each column:")
display(df.isnull().sum().to_frame("Missing Values"))

print("Total missing values:", int(df.isnull().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))

if df.isnull().sum().sum() == 0:
    print("\nResult: No missing values found.")
else:
    print("\nResult: Missing values were found and need to be handled.")


## 5. Exploratory Data Analysis (EDA)

In [ ]:
# Distribution of all numerical variables
df.hist(figsize=(14, 10), bins=30, edgecolor="black")
plt.suptitle("Distribution of California Housing Features", fontsize=16)
plt.tight_layout()
plt.show()


In [ ]:
# Correlation matrix
correlation = df.corr(numeric_only=True)

plt.figure(figsize=(10, 7))
sns.heatmap(correlation, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()


In [ ]:
# Median income vs house value
plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=df,
    x="MedInc",
    y="MedHouseVal",
    alpha=0.25
)
plt.title("Median Income vs Median House Value")
plt.xlabel("Median Income")
plt.ylabel("Median House Value")
plt.tight_layout()
plt.show()


### EDA observations

- The dataset contains **20,640 records** and **8 input features**.
- The target variable is continuous, so this is a regression problem.
- Median income (`MedInc`) has a strong positive relationship with median house value.
- Latitude and longitude also provide useful geographic information.
- The missing-value check is performed before model training.
- Linear Regression is used as a simple baseline model.


## 6. Select Features and Target

In [ ]:
X = df.drop(columns=["MedHouseVal"])
y = df["MedHouseVal"]

print("Feature columns:")
print(list(X.columns))

print("\nX shape:", X.shape)
print("y shape:", y.shape)


## 7. Split Data into Training and Testing Sets

The task specifies a train/test split. Here, **80%** of the data is used for training and **20%** for testing.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))
print("Training percentage:", len(X_train) / len(df) * 100)
print("Testing percentage:", len(X_test) / len(df) * 100)


## 8. Train the Linear Regression Model

In [ ]:
model = LinearRegression()

# Train the model
model.fit(X_train, y_train)

# Predict values for the test set
y_pred = model.predict(X_test)

print("Linear Regression model trained successfully.")
print("Intercept:", model.intercept_)


## 9. Evaluate the Model – MAE, RMSE and R²

- **MAE:** Mean Absolute Error. Lower values indicate smaller average errors.
- **RMSE:** Root Mean Squared Error. Lower values indicate smaller errors and it gives greater weight to large errors.
- **R²:** R-squared. Higher values indicate that the model explains more variation in the target.

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

metrics = pd.DataFrame({
    "Metric": ["MAE", "RMSE", "R²"],
    "Value": [mae, rmse, r2]
})

display(metrics)

print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")


## 10. Actual vs Predicted House Values

In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(
    y_test,
    y_pred,
    alpha=0.35,
    label="Predictions"
)

# Perfect-prediction reference line
min_value = min(y_test.min(), y_pred.min())
max_value = max(y_test.max(), y_pred.max())

plt.plot(
    [min_value, max_value],
    [min_value, max_value],
    "r--",
    linewidth=2,
    label="Perfect prediction"
)

plt.xlabel("Actual House Value")
plt.ylabel("Predicted House Value")
plt.title("Actual vs Predicted House Values")
plt.legend()
plt.tight_layout()
plt.show()


## 11. Residual Analysis

A residual is the difference between the actual value and the predicted value.

In [ ]:
residuals = y_test - y_pred

plt.figure(figsize=(8, 5))
plt.scatter(
    y_pred,
    residuals,
    alpha=0.35
)
plt.axhline(0, color="red", linestyle="--", linewidth=2)

plt.xlabel("Predicted House Value")
plt.ylabel("Residual (Actual - Predicted)")
plt.title("Residual Plot")
plt.tight_layout()
plt.show()


In [ ]:
print("Residual summary:")
display(pd.Series(residuals).describe().round(4))


## 12. Feature Coefficients

The coefficients show the direction and magnitude of the Linear Regression relationship for each feature, while keeping the other features fixed.

In [ ]:
coefficients = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_
})

coefficients["Absolute_Coefficient"] = coefficients["Coefficient"].abs()
coefficients = coefficients.sort_values(
    "Absolute_Coefficient",
    ascending=False
).drop(columns=["Absolute_Coefficient"])

display(coefficients)


## 13. Save the Trained Model

The task optionally allows a saved model. The trained Linear Regression model is saved as a `.pkl` file.

In [ ]:
model_filename = "california_house_linear_regression.pkl"

with open(model_filename, "wb") as file:
    pickle.dump(model, file)

print("Model saved successfully.")
print("File:", os.path.abspath(model_filename))


## 14. Load the Saved Model and Verify It

In [ ]:
with open(model_filename, "rb") as file:
    loaded_model = pickle.load(file)

loaded_prediction = loaded_model.predict(X_test)

print("Saved model loaded successfully.")
print("Predictions from saved model match original model:",
      np.allclose(y_pred, loaded_prediction))


## 15. Example Prediction on New Input

The following example uses the same eight input features as the training dataset. The prediction is converted from the dataset's $100,000 unit into an approximate dollar value.

In [ ]:
sample_house = pd.DataFrame([{
    "MedInc": 5.0,
    "HouseAge": 25.0,
    "AveRooms": 5.5,
    "AveBedrms": 1.1,
    "Population": 1200.0,
    "AveOccup": 3.0,
    "Latitude": 34.0,
    "Longitude": -118.0
}])

prediction_units = loaded_model.predict(sample_house)[0]
prediction_dollars = prediction_units * 100000

print(f"Predicted median house value: ${prediction_dollars:,.2f}")


## 16. Final Results Summary

The table below provides the final evaluation values calculated by the trained model.

In [ ]:
results = pd.DataFrame({
    "Metric": ["MAE", "RMSE", "R²"],
    "Result": [mae, rmse, r2]
})

display(results.style.format({"Result": "{:.4f}"}))


## 17. Conclusion

This project successfully implements an end-to-end machine-learning workflow for house price prediction using the California Housing dataset.

### Work completed
- Loaded the California Housing dataset using scikit-learn.
- Explored the data using descriptive statistics and visualizations.
- Checked missing values and duplicate records.
- Performed correlation analysis.
- Split the data into training and testing sets.
- Trained a Linear Regression model.
- Evaluated the model using **MAE, RMSE and R²**.
- Created **Actual vs Predicted** and **Residual** plots.
- Examined model feature coefficients.
- Saved and reloaded the trained model.
- Demonstrated a prediction on a new input.

### Possible Improvements
1. Compare Linear Regression with Ridge and Lasso Regression.
2. Try tree-based models such as Random Forest or Gradient Boosting.
3. Use cross-validation for a more robust evaluation.
4. Perform deeper outlier and residual analysis.
5. Build a small user interface for entering house information and receiving predictions.


# Submission Checklist

According to the internship task, the main required deliverable is a **Jupyter Notebook (.ipynb) containing code, plots and comments**.

This single notebook contains:
- ✅ Data loading
- ✅ EDA
- ✅ Missing-value check
- ✅ Train/test split
- ✅ Linear Regression
- ✅ MAE
- ✅ RMSE
- ✅ R²
- ✅ Actual vs Predicted plot
- ✅ Residual plot
- ✅ Model saving
- ✅ Example prediction
- ✅ Findings and improvement ideas
- ✅ Short report-style documentation

**Before submitting:** use **Run All → Save** so the notebook contains all generated outputs.
